In [2]:
import tinycudann as tcnn
import torch

from multires_hash_encoding import MultiresEncodingConfig, MultiresEncoding

import time

p:\Programs\miniconda3\envs\master-pytorch\Lib\site-packages\tinycudann\modules.py:58: UserWarning: tinycudann was built for lower compute capability (52) than the system's (86). Performance may be suboptimal.
  warnings.warn(f"tinycudann was built for lower compute capability ({cc}) than the system's ({system_compute_capability}). Performance may be suboptimal.")


In [3]:
device = 'cuda'

config = {
		"otype": "HashGrid",
		"n_levels": 8,
		"n_features_per_level": 2,
		"log2_hashmap_size": 15,
		"base_resolution": 16,
		"per_level_scale": 1.5,
		"fixed_point_pos": False
	}

tcnn_enc = tcnn.Encoding(2, config)
tcnn_enc.to(device)

conf = MultiresEncodingConfig()

conf.nlevels = 8
conf.minres = (16, 16)
conf.maxres = (512, 512)

py_enc = MultiresEncoding(**vars(conf), dtype=torch.float32, device = device)


In [4]:
num_samples = 1000000

t = torch.rand((num_samples, 2), device = device)

start = time.time()
out = py_enc(t)
py_enc_time = time.time() - start

start = time.time()
out = tcnn_enc(t)
tcnn_enc_time = time.time() - start

print(f"py enc encoded {num_samples} in {py_enc_time}")
print(f"tcnn enc encoded {num_samples} in {tcnn_enc_time}")

py enc encoded 1000000 in 0.21030759811401367
tcnn enc encoded 1000000 in 0.022609233856201172


In [24]:
enc_config = {
    "otype": "OneBlob", # Component type.
    "n_bins": 16        # Number of bins per encoded dimension.
}
enc = tcnn.Encoding(2, enc_config)

net_config = {
    "otype": "FullyFusedMLP",    # Component type.
    "activation": "ReLU",        # Activation of hidden layers.
    "output_activation": "None", # Activation of the output layer.
    "n_neurons": 32,            # Neurons in each hidden layer.
                                # May only be 16, 32, 64, or 128.
    "n_hidden_layers": 2,        # Number of hidden layers.
}

K = 3

network = tcnn.Network(enc.n_output_dims, (3 * K  - 1), net_config)
network2 = tcnn.NetworkWithInputEncoding(1, (3 * K  - 1), enc_config, net_config)

t = torch.rand((16, 1), device = 'cuda')

network2(t)

tensor([[-0.0745,  0.0520,  0.0857, -0.1000,  0.0194,  0.1941, -0.0644, -0.0930],
        [-0.0528,  0.0272,  0.0185, -0.0909, -0.0078,  0.1905, -0.0968, -0.1082],
        [-0.0110, -0.0766, -0.0140, -0.0319,  0.0885,  0.0914, -0.0594, -0.0354],
        [-0.0749,  0.0135,  0.2115, -0.0914,  0.0011,  0.2407, -0.1873, -0.0651],
        [-0.1709, -0.0363,  0.0675, -0.0240,  0.0706,  0.0897,  0.0242, -0.0723],
        [-0.0331, -0.0004,  0.0014, -0.0736,  0.0342,  0.1850, -0.0890, -0.1356],
        [-0.0261, -0.0046, -0.0336, -0.0254, -0.0031,  0.0085, -0.0996, -0.0006],
        [-0.0485,  0.0253,  0.0474, -0.0721,  0.0456,  0.1911, -0.0857, -0.1121],
        [-0.0773,  0.0138,  0.0050,  0.0227,  0.0095,  0.1715, -0.1349,  0.0854],
        [-0.0519,  0.0406,  0.0679, -0.0803,  0.0259,  0.2002, -0.0691, -0.1092],
        [-0.0198, -0.0081,  0.0124, -0.0585,  0.0398,  0.1904, -0.0735, -0.1299],
        [-0.0703,  0.0115,  0.2183, -0.0897, -0.0033,  0.2406, -0.1871, -0.0631],
        [-0.0579